# 배리어프리 접근성 분석 화면
사진을 업로드하면 가짜 탐지 박스와 A/B/C 접근성 등급을 출력합니다.

In [2]:
!pip -q install -U gradio pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 which is incompatible.


In [3]:
import gradio as gr
from PIL import Image

print("Gradio 버전:", gr.__version__)
print("설치 정상 완료")

Gradio 버전: 5.50.0
설치 정상 완료


In [4]:
%pip uninstall -y pillow
%pip install --no-cache-dir --force-reinstall "pillow==11.3.0"

Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 72.6 MB/s eta 0:00:00


In [5]:
import PIL
import gradio as gr
from PIL import Image, ImageDraw, ImageFont

print("Pillow 버전:", PIL.__version__)
print("Gradio 버전:", gr.__version__)
print("정상적으로 불러왔습니다.")

Pillow 버전: 11.3.0
Gradio 버전: 5.50.0
정상적으로 불러왔습니다.


In [1]:
import gradio as gr
from PIL import Image, ImageDraw, ImageFont


def analyze_accessibility(image, demo_case):
    """UI 확인용 가짜 탐지 결과를 생성합니다."""
    if image is None:
        raise gr.Error("먼저 입구 사진을 업로드해 주세요.")

    result_image = image.convert("RGB").copy()
    draw = ImageDraw.Draw(result_image)

    width, height = result_image.size
    line_width = max(3, int(min(width, height) * 0.008))
    font = ImageFont.load_default()

    stairs_detected = False
    ramp_detected = False

    # 실제 YOLO 모델이 완성되면 이 부분을 model.predict() 결과로 교체합니다.
    if demo_case == "A등급: 경사로 있음 / 계단 없음":
        ramp_detected = True
        boxes = [
            ("ramp", (int(width * 0.18), int(height * 0.55),
                      int(width * 0.86), int(height * 0.90)), "#22c55e")
        ]
    elif demo_case == "A등급: 계단 없음 / 경사로 없음":
        boxes = []
    elif demo_case == "B등급: 계단 있음 / 경사로 있음":
        stairs_detected = True
        ramp_detected = True
        boxes = [
            ("stairs", (int(width * 0.08), int(height * 0.50),
                        int(width * 0.48), int(height * 0.90)), "#ef4444"),
            ("ramp", (int(width * 0.52), int(height * 0.48),
                      int(width * 0.92), int(height * 0.90)), "#22c55e")
        ]
    else:
        stairs_detected = True
        boxes = [
            ("stairs", (int(width * 0.16), int(height * 0.48),
                        int(width * 0.86), int(height * 0.90)), "#ef4444")
        ]

    for label, box, color in boxes:
        draw.rectangle(box, outline=color, width=line_width)
        text_box = draw.textbbox((0, 0), label, font=font)
        text_width = text_box[2] - text_box[0]
        text_height = text_box[3] - text_box[1]
        label_x = box[0]
        label_y = max(0, box[1] - text_height - 10)
        draw.rectangle(
            (label_x, label_y, label_x + text_width + 12, label_y + text_height + 8),
            fill=color,
        )
        draw.text((label_x + 6, label_y + 4), label, fill="white", font=font)

    if not stairs_detected:
        grade = "A"
        meaning = "휠체어·유모차 접근이 비교적 쉬움"
        grade_color = "#16a34a"
        grade_bg = "#dcfce7"
    elif stairs_detected and ramp_detected:
        grade = "B"
        meaning = "경사로로 우회할 수 있으나 일부 주의가 필요함"
        grade_color = "#ca8a04"
        grade_bg = "#fef9c3"
    else:
        grade = "C"
        meaning = "계단만 있어 접근이 어렵거나 보조가 필요함"
        grade_color = "#dc2626"
        grade_bg = "#fee2e2"

    grade_card = f"""
    <div class="grade-card" style="border:2px solid {grade_color}; background:{grade_bg};">
        <div class="grade-label">접근성 분석 결과</div>
        <div class="grade-value" style="color:{grade_color};">{grade}등급</div>
        <div class="grade-meaning">{meaning}</div>
    </div>
    """

    details = f"""
### 탐지 결과

- **계단:** {"탐지됨" if stairs_detected else "탐지되지 않음"}
- **경사로:** {"탐지됨" if ramp_detected else "탐지되지 않음"}
- **최종 등급:** **{grade}등급**
- **판정 내용:** {meaning}

> 현재 화면은 UI 확인을 위한 가짜 탐지값을 사용하고 있습니다. 실제 모델 학습 후 YOLO 탐지 결과와 연결하면 됩니다.
"""

    return result_image, grade_card, details


def reset_screen():
    return None, None, """
    <div class="empty-card">사진을 업로드한 뒤 <b>접근성 분석하기</b> 버튼을 눌러 주세요.</div>
    """, "탐지 결과가 여기에 표시됩니다."


CSS = """
.gradio-container {max-width:1120px !important; margin:0 auto !important;}
#main-title {text-align:center; margin-bottom:4px;}
#sub-title {text-align:center; color:#64748b; margin-bottom:20px;}
.grade-card {border-radius:18px; padding:26px 20px; text-align:center; min-height:180px; display:flex; flex-direction:column; justify-content:center;}
.grade-label {font-size:15px; font-weight:700; color:#475569; margin-bottom:8px;}
.grade-value {font-size:52px; line-height:1.1; font-weight:900; margin-bottom:12px;}
.grade-meaning {font-size:17px; font-weight:700; color:#1e293b;}
.empty-card {border:2px dashed #cbd5e1; border-radius:18px; padding:48px 20px; text-align:center; color:#64748b; min-height:180px; display:flex; align-items:center; justify-content:center;}
#analyze-button {background:#4f46e5; color:white; border:none;}
"""


with gr.Blocks(css=CSS, title="배리어프리 접근성 분석") as demo:
    gr.Markdown("# Vision AI 배리어프리 접근성 분석", elem_id="main-title")
    gr.Markdown(
        "가게 입구 사진을 업로드하면 계단·경사로 탐지 결과와 접근성 등급을 보여줍니다.",
        elem_id="sub-title",
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="pil", label="① 입구 사진 업로드", height=420)

            with gr.Accordion("개발자용 임시 설정", open=False):
                demo_case = gr.Radio(
                    choices=[
                        "A등급: 경사로 있음 / 계단 없음",
                        "A등급: 계단 없음 / 경사로 없음",
                        "B등급: 계단 있음 / 경사로 있음",
                        "C등급: 계단 있음 / 경사로 없음",
                    ],
                    value="C등급: 계단 있음 / 경사로 없음",
                    label="가짜 탐지 시나리오",
                )

            with gr.Row():
                analyze_button = gr.Button("② 접근성 분석하기", variant="primary", elem_id="analyze-button")
                reset_button = gr.Button("초기화")

        with gr.Column(scale=1):
            output_image = gr.Image(type="pil", label="③ 객체 탐지 결과", height=420, interactive=False)

    with gr.Row():
        with gr.Column(scale=1):
            grade_output = gr.HTML(
                '<div class="empty-card">사진을 업로드한 뒤 <b>접근성 분석하기</b> 버튼을 눌러 주세요.</div>'
            )
        with gr.Column(scale=1):
            detail_output = gr.Markdown("탐지 결과가 여기에 표시됩니다.")

    analyze_button.click(
        fn=analyze_accessibility,
        inputs=[input_image, demo_case],
        outputs=[output_image, grade_output, detail_output],
    )

    reset_button.click(
        fn=reset_screen,
        inputs=[],
        outputs=[input_image, output_image, grade_output, detail_output],
    )


demo.launch(share=True)


/tmp/ipykernel_5248/2459922550.py:115: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="배리어프리 접근성 분석") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc087ca8f8d980cb7a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
